# P4 · Probe: TINS on our features (does ReNeg on top of TINS beat TINS?)

TINS (arXiv 2605.10756, May 2026) is the strongest published method on Four-OOD. It uses labelled ImageNet
images (16 per class; its paper shows no gain beyond 4) and, at test time, turns images that look OOD into text
embeddings with 30 gradient steps through CLIP's text encoder. `reneg/tins.py` re-implements its official code on
our feature cache. This notebook runs TINS on OpenOOD v1.5 and Four-OOD (three stream orders) and saves, for every
image, TINS's score and whether it was inverted. With those files I replay ReNeg on top of TINS locally, so the
combination can be tested without more GPU runs.

* ID prototypes: 5 labelled images per class, OpenOOD's ImageNet ID-val split. OpenOOD v1.5's test split does not
  contain them; Four-OOD's 50k ID set does, so the summary also scores Four-OOD without them (`four_ood_45k`).
* GPU needed (any CUDA GPU; less than 8 GB of memory is handled by inverting fewer images at a time). The cell
  after the set-up times TINS on one stream and prints an estimate for the whole run. Results are saved per
  stream; after a dropped session, run all cells again and finished streams are skipped.

**Output:** `MyDrive/ReNeg/transfers/p4_tins/`: `tins_runs.csv`, `tins_summary.csv`, `shot_mask.npy` and
`tins_traces_*.zip` (attach all of them).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("P4_tins_probe", need_gpu=True)
log = ctx.log
import json, time, glob, zipfile
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from oodlab.runner import Runner
from oodlab.metrics import openood_metrics
from oodlab.clipwrap import load_clip
from reneg.tins import TINSConfig, TINSFeat, build_init_candidates, build_static_negatives
from reneg.transport import l2n
cache = find_cache(ctx, "ViT-B/16")
bank = cache.load_textbank()
runner = Runner(cache, device=ctx.device, logger=log)
OUT = f"{DRIVE_ROOT}/transfers/p4_tins"
TR = os.path.join(OUT, "tins_traces")
os.makedirs(TR, exist_ok=True)
SEEDS = [0, 1, 2]
PROTOCOLS = [p for p in ("openood_v15", "four_ood") if not runner.missing(p)]
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip("ViT-B/16", device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS,
                  random_init=os.environ.get("RENEG_FAKE_GEN") == "1")
REAL = hasattr(model, "token_embedding") and "imagenet_val" in cache.available()
print("device:", ctx.device, "| protocols:", PROTOCOLS, "| real CLIP text encoder:", REAL)
if REAL and ctx.device == "cpu":
    print("WARNING: no GPU. TINS inverts images with 30 gradient steps each; on a CPU this takes days.")

## ID prototypes (5 per class) and TINS's static negatives (a few minutes, cached)

In [ ]:
static_path = os.path.join(OUT, "tins_static.pt")
shot_path = os.path.join(OUT, "shot_mask.npy")
if REAL:
    v5 = runner.get("imagenet_val")
    X5, y5 = l2n(v5.feats.float()), torch.as_tensor(np.asarray(v5.labels)).long()
    C = len(bank.id_names)
    cnt = torch.bincount(y5, minlength=C)
    proto = l2n(torch.zeros(C, X5.shape[1]).index_add_(0, y5, X5))
    if (cnt == 0).any():                     # TINS needs one prototype per class (the official code stops here)
        print(f"WARNING: {int((cnt == 0).sum())} classes have no labelled image: their text embedding is used")
        proto[cnt == 0] = l2n(bank.id_text.float())[cnt == 0]
    acc = float(((X5 @ l2n(bank.id_text.float()).T).argmax(1) == y5).float().mean() * 100)
    print(f"{len(X5)} labelled ID images, {int((cnt > 0).sum())} classes (min {int(cnt.min())} per class); "
          f"zero-shot accuracy on them {acc:.1f}% (about 70% means the labels line up)")
    if "imagenet_val_all" in cache.available() and not os.path.isfile(shot_path):
        A = l2n(runner.get("imagenet_val_all").feats.float().to(ctx.device)); B = X5.to(ctx.device)
        shot = torch.zeros(len(A), dtype=torch.bool)
        for s in range(0, len(A), 4096):
            shot[s:s + 4096] = ((A[s:s + 4096] @ B.T).max(1).values > 0.999).cpu()
        np.save(shot_path, shot.numpy())
        del A, B
    if os.path.isfile(shot_path):
        print("labelled images found inside Four-OOD's 50k ID set:", int(np.load(shot_path).sum()), "(expect about 5,000)")
    if os.path.isfile(static_path):
        st = torch.load(static_path, weights_only=False)
    else:
        t0 = time.time()
        neg, words = build_static_negatives(model, bank.id_text.float(), proto, n=2000,
                                            positive_labels=list(bank.id_names), log=print)
        st = {"neg": neg, "words": words, "cands": build_init_candidates(model, words, proto), "proto": proto}
        torch.save(st, static_path)
        print(f"built in {(time.time() - t0) / 60:.1f} min")
    print(len(st["words"]), "static negatives, e.g.", st["words"][:8])
else:
    print("test stub of oodlab (no CLIP text encoder): the TINS cells are skipped")

## Timing check on one stream (NINCO, stream order 0) and an estimate for the whole run

In [ ]:
def make_tins():
    return TINSFeat(model, bank.id_text.float(), st["proto"], st["neg"], st["cands"], TINSConfig())

class TraceHook:
    """Saves, per image in stream order: TINS's score, its score before the batch's own inversions, whether it
    was a candidate and whether its inverted text entered the bank, plus the stream rows (for alignment)."""
    def __init__(self, path):
        self.path, self.parts = path, []
    def on_reset(self, method, seg):
        self.parts = []
    def on_step(self, method, seg, start, end, out, stream):
        i = out.info
        self.parts.append((start, out.score.float().cpu().numpy(), i["pre"].numpy(), i["cand"].numpy(),
                           i["kept"].numpy()))
    def on_end(self, method, stream):
        n = len(stream)
        arr = {k: np.zeros(n, dt) for k, dt in (("score", np.float32), ("pre", np.float32), ("cand", bool),
                                                ("kept", bool))}
        for s, sc, pre, cand, kept in self.parts:
            for k, a in (("score", sc), ("pre", pre), ("cand", cand), ("kept", kept)):
                arr[k][s:s + len(a)] = a
        tmp = self.path + ".tmp.npz"
        np.savez_compressed(tmp, is_ood=(np.asarray(stream.labels) == -1), rows=stream.rows.astype(np.int32),
                            n_inv=method.n_inv, t_inv=method.t_inv, **arr)
        os.replace(tmp, self.path)

def trace_path(protocol, ds, seed):
    return os.path.join(TR, f"{protocol}__{ds}__s{seed}.npz")

runs_path = os.path.join(OUT, "tins_runs.csv")

def run_one(protocol, ds, seed):
    global done
    if os.path.isfile(trace_path(protocol, ds, seed)):
        return None
    t0 = time.time()
    df = runner.evaluate(make_tins(), protocol, seeds=[seed], datasets=[ds], config="tins_5shot",
                         hooks_factory=lambda name, sd: [TraceHook(trace_path(protocol, name, sd))])
    tr = np.load(trace_path(protocol, ds, seed))
    df["label"] = "tins_5shot"
    df["n_inverted"] = int(tr["n_inv"])
    df["inversion_seconds"] = float(tr["t_inv"])
    done = pd.concat([done, df], ignore_index=True)
    done.to_csv(runs_path, index=False)
    r = df.iloc[0]
    print(f"{protocol} {ds} s{seed}: FPR95 {r.fpr95:.2f} (ID-positive {r.fpr95_idpos:.2f}), AUROC {r.auroc:.2f}, "
          f"{r.n_inverted} images inverted, {(time.time() - t0) / 60:.1f} min", flush=True)
    return r

SETS = {"openood_v15": (["ssb_hard", "ninco", "inaturalist", "textures", "openimage_o"], "imagenet_test"),
        "four_ood": (["inaturalist", "sun", "places", "textures_all"], "imagenet_val_all")}
done = pd.read_csv(runs_path) if os.path.isfile(runs_path) else pd.DataFrame()
if REAL and "openood_v15" in PROTOCOLS:
    run_one("openood_v15", "ninco", 0)
sel = done[(done.protocol == "openood_v15") & (done.dataset == "ninco") & (done.seed == 0)] if len(done) else done
if REAL and len(sel):
    r = sel.iloc[-1]
    tr = np.load(trace_path("openood_v15", "ninco", 0))
    n = len(tr["score"])
    per_img = (r.method_seconds - r.inversion_seconds) / n                 # scoring, per image
    per_inv = r.inversion_seconds / max(1, r.n_inverted)                   # inversion, per candidate image
    f_id = float(tr["cand"][~tr["is_ood"]].mean())                         # ID images that become candidates
    hours = 0.0
    for p in PROTOCOLS:
        n_id = len(runner.get(SETS[p][1]))
        for ds in SETS[p][0]:
            n_ood = len(runner.get(ds))                                    # upper bound: every OOD image inverted
            hours += ((n_ood + n_id) * per_img + (n_ood + f_id * n_id) * per_inv) / 3600
    print(f"NINCO: {per_inv * 1000:.1f} ms per inverted image, {100 * f_id:.1f}% of ID images inverted, "
          f"{100 * tr['cand'][tr['is_ood']].mean():.1f}% of OOD images")
    print(f"estimate for everything: at most {len(SEEDS) * hours:.1f} hours ({hours:.1f} h per stream order)")

## Run TINS on every stream (stream order 0 for all sets first, then 1 and 2; saved per stream)

In [ ]:
if REAL:
    for seed in SEEDS:
        for protocol in PROTOCOLS:
            for ds in SETS[protocol][0]:
                run_one(protocol, ds, seed)

## Summary from the saved traces (both FPR95 conventions), and the files to attach

`fpr95` is OpenOOD's convention (OOD positive; what TANL reports and our tables use). `fpr95_idpos` is the
convention of TINS's code (ID positive: OOD images accepted when 95% of ID images are kept). TINS's paper
(16 labelled images per class, ID positive): Four-OOD 6.72 (iNaturalist 0.21, SUN 3.84, Places 12.73, Textures
10.09), OpenOOD v1.5 near-OOD 57.88, far-OOD 12.99.

In [ ]:
def TRACES():
    return sorted(f for f in glob.glob(os.path.join(TR, "*.npz")) if not f.endswith(".tmp.npz"))

shot = np.load(shot_path) if os.path.isfile(shot_path) else None
rows = []
for f in TRACES():
    protocol, ds, s = os.path.basename(f)[:-4].split("__")
    tr = np.load(f)
    lab = np.where(tr["is_ood"], -1, 0)
    variants = [(protocol, np.ones(len(lab), bool))]
    if protocol == "four_ood" and shot is not None:
        n_ood = len(runner.get(ds))
        idr = tr["rows"].astype(np.int64) - n_ood                          # ID row, or negative for OOD images
        keep = np.ones(len(lab), bool)
        keep[idr >= 0] = ~shot[idr[idr >= 0]]
        variants.append(("four_ood_45k", keep))
    for name, keep in variants:
        m = openood_metrics(tr["score"][keep], lab[keep])
        rows.append({"protocol": name, "dataset": ds, "seed": int(s[1:]), "fpr95": m["fpr95"],
                     "fpr95_idpos": m["fpr95_idpos"], "auroc": m["auroc"], "inverted": int(tr["cand"].sum()),
                     "kept": int(tr["kept"].sum())})
if rows:
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT, "tins_summary.csv"), index=False)
    per = df.groupby(["protocol", "dataset"])[["fpr95", "fpr95_idpos", "auroc"]].agg(["mean", "std"]).round(2)
    pd.set_option("display.width", 200)
    print(per.to_string())
    groups = {"near": ["ssb_hard", "ninco"], "far": ["inaturalist", "textures", "openimage_o"]}
    mean = df.groupby(["protocol", "dataset"])[["fpr95", "fpr95_idpos", "auroc"]].mean()
    for p, sets in [("openood_v15", groups["near"]), ("openood_v15", groups["far"]),
                    ("four_ood", SETS["four_ood"][0]), ("four_ood_45k", SETS["four_ood"][0])]:
        try:
            g = mean.loc[[(p, d) for d in sets]].mean()
            print(f"{p} {sets}: FPR95 {g.fpr95:.2f} | ID-positive {g.fpr95_idpos:.2f} | AUROC {g.auroc:.2f}")
        except KeyError:
            pass
parts, cur, size = [], [], 0
for f in TRACES():
    if cur and size + os.path.getsize(f) > 22e6:
        parts.append(cur); cur, size = [], 0
    cur.append(f); size += os.path.getsize(f)
if cur:
    parts.append(cur)
for old in glob.glob(os.path.join(OUT, "tins_traces_*.zip")):
    os.remove(old)
for k, fs in enumerate(parts, 1):
    with zipfile.ZipFile(os.path.join(OUT, f"tins_traces_{k}.zip"), "w") as z:
        for f in fs:
            z.write(f, os.path.basename(f))
print("\nresult files:")
for f in [runs_path, os.path.join(OUT, "tins_summary.csv"), shot_path] + sorted(glob.glob(os.path.join(OUT, "tins_traces_*.zip"))):
    if os.path.isfile(f):
        print("  ", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")
finish(ctx, {"traces": len(TRACES())})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")